# Auswertung Ziele


In [1]:
import os; print(os.environ.get("LD_LIBRARY_PATH"))

/opt/ohpc/pub/mpi/libfabric/1.19.0/lib:/opt/ohpc/pub/mpi/ucx-ohpc/1.15.0/lib:/opt/ohpc/pub/libs/hwloc/lib:/opt/ohpc/pub/mpi/openmpi4-gnu9/4.1.1/lib:/opt/ohpc/pub/compiler/gcc/9.4.0/lib64


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import json
import os

import testing_different_goals_framework as tdgf


results_folder = sorted(glob.glob("output/nsgaii_testing*"))[-1]     
print("results from", results_folder)
per_seed_results = pd.read_csv(f"{results_folder}/per_seed.csv")
mean_per_strategy = per_seed_results.groupby(["generation", "strategy"])[tdgf.objective_candidates].mean()
pareto_front = pd.read_csv(f"{results_folder}/front.csv")     
objectives = json.load(open(f"{results_folder}/run_settings.json"))["objectives"]  

/opt/ohpc/pub/spack/develop/opt/spack/linux-rocky8-zen2/gcc-12.2.0/boost-1.85.0-qz55jyxhn3c3vfverplcn7u2obcdz3nz/lib:$LD_LIBRARY_PATH


ImportError: libboost_filesystem.so.1.85.0: cannot open shared object file: No such file or directory

In [ ]:
rank_correlation = mean_per_strategy.loc[0].corr(method="spearman")
figure, axis = plt.subplots(figsize=(8, 7))
axis.imshow(rank_correlation, cmap="RdBu_r", vmin=-1, vmax=1)
axis.set_xticks(range(len(rank_correlation)), rank_correlation.columns, rotation=90)
axis.set_yticks(range(len(rank_correlation)), rank_correlation.columns)
for (row, column), value in np.ndenumerate(rank_correlation.to_numpy()):
    axis.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=7)
plt.show()

TO DO

In [ ]:
best_so_far = mean_per_strategy[objectives].groupby("generation").min().cummin()
best_so_far.plot(subplots=True, layout=(1, len(objectives)), figsize=(4.5 * len(objectives), 3), marker=".")
pd.plotting.scatter_matrix(pareto_front[objectives], figsize=(7, 7))
plt.show()

In [ ]:
decoded_strategies = []
for _, strategy in pareto_front[tdgf.variable_names].iterrows():
    (first_change_day, second_change_day), periods = tdgf.decode_strategy(strategy)
    decoded_strategies.append({"change_day_1": first_change_day, "change_day_2": second_change_day,
                               **{f"{name}_{period_number}": value
                                  for period_number, period in enumerate(periods, 1)
                                  for name, value in period.items()}})
decoded_strategies = pd.DataFrame(decoded_strategies).round(2)
pd.concat([pareto_front[objectives], decoded_strategies], axis=1).sort_values(objectives[0])